# Chapter 8 — Determinants: SageMath Companion

Symbolic, exact-arithmetic counterpart to `code/python/08_determinants.ipynb`. We use:

- `Matrix(QQ, ...)` for exact rational matrices (row operations, Cramer's Rule, adjugate).
- `PolynomialRing(QQ, 'L')` for **characteristic polynomials** with integer/rational coefficients.
- `SR` (symbolic ring) for the Vandermonde identity with variable entries.

Everything below is exact. No floating-point roundoff, no `1.99999 ≈ 2` theater.

**To run this notebook:** open in [CoCalc](https://cocalc.com/) (no install needed), or locally with the SageMath kernel.

**Kernel:** SageMath.

**Layout:**

1. 2×2 and 3×3 `det` by three methods — hand formula, cofactor recursion, row reduction.
2. Cofactor expansion along any row/column — implement from scratch.
3. Effect of elementary row operations on `det` — verify the three rules.
4. Multiplicativity `det(AB) = det(A) det(B)` over `QQ`.
5. `det(Aᵀ) = det(A)` and block triangular determinants.
6. Symbolic Vandermonde: derive `(b−a)(c−a)(c−b)` directly.
7. Cramer's Rule on a 3×3 — matches `solve_right`.
8. Adjugate formula: `A · adj(A) = (det A) · I`, and the closed-form inverse.
9. Characteristic polynomial `det(A − λI)` — preview of eigenvalues (Ch 9).
10. **Exercises** with solutions.


## 1. Three ways to compute `det` for a 3×3

Anchor matrix from the notes: `A = [[2,1,3],[1,4,2],[3,2,5]]`. Expected: `det A = 3`.

In [ ]:
A = Matrix(QQ, [[2, 1, 3],
                 [1, 4, 2],
                 [3, 2, 5]])

# (a) Sarrus rule
sarrus = (
    A[0,0]*A[1,1]*A[2,2] + A[0,1]*A[1,2]*A[2,0] + A[0,2]*A[1,0]*A[2,1]
    - A[0,2]*A[1,1]*A[2,0] - A[0,0]*A[1,2]*A[2,1] - A[0,1]*A[1,0]*A[2,2]
)
show('Sarrus:', sarrus)

# (b) Sage's built-in .det()
show('A.det():', A.det())

# (c) Row reduction to upper triangular via LU factorization.
P, L, U = A.LU()
show('P ='); show(P)
show('L ='); show(L)
show('U ='); show(U)
print('det A via LU = P.det() · prod(diag U) =', P.det() * prod(U[i, i] for i in range(3)))

## 2. Cofactor expansion from scratch

Implement the Laplace expansion recursively. Cross-check on several matrices.

In [ ]:
def det_cofactor(M):
    '''det via cofactor (Laplace) expansion along row 0.'''
    n = M.nrows()
    if n == 1:
        return M[0, 0]
    if n == 2:
        return M[0, 0]*M[1, 1] - M[0, 1]*M[1, 0]
    total = 0
    for j in range(n):
        minor = M.delete_rows([0]).delete_columns([j])
        total += (-1)**j * M[0, j] * det_cofactor(minor)
    return total


tests = [
    Matrix(QQ, [[3, 1], [1, 2]]),
    Matrix(QQ, [[2, 1, 3], [1, 4, 2], [3, 2, 5]]),
    Matrix(QQ, [[1, 2, 0, 0], [0, 3, 0, 0], [4, 5, 6, 0], [7, 8, 9, 2]]),
]

for M in tests:
    mine, sage_val = det_cofactor(M), M.det()
    print(f'cofactor = {mine}  sage = {sage_val}  match = {mine == sage_val}')

## 3. Elementary row operations and their effect on `det`

Three operations, three rules — verify each.

In [ ]:
A = Matrix(QQ, [[2, 1, 3], [1, 4, 2], [3, 2, 5]])
d0 = A.det()
print(f'det A                             = {d0}')

# (a) swap rows 0 and 1
B = A.with_swapped_rows(0, 1)
print(f'det after swap rows 0, 1          = {B.det()}   (expected {-d0})')

# (b) scale row 1 by 5
C = A.with_rescaled_row(1, 5)
print(f'det after scaling row 1 by 5      = {C.det()}   (expected {5*d0})')

# (c) add 2·row 0 to row 2
D = A.with_added_multiple_of_row(2, 0, 2)
print(f'det after adding 2·row 0 to row 2 = {D.det()}   (expected {d0})')

# (d) transpose — same det
print(f'det A^T                           = {A.transpose().det()}   (expected {d0})')

# (e) det(3A) for 3x3 = 27 · det A
print(f'det(3A)                           = {(3*A).det()}   (expected {27*d0})')

## 4. Multiplicativity over `QQ`

`det(AB) = det(A) · det(B)`. Sample 50 random rational matrices and verify.

In [ ]:
from sage.misc.prandom import randint

passes, fails = 0, 0
for _ in range(50):
    M1 = random_matrix(QQ, 5, algorithm='echelonizable', rank=5)
    M2 = random_matrix(QQ, 5, algorithm='echelonizable', rank=5)
    if (M1 * M2).det() == M1.det() * M2.det():
        passes += 1
    else:
        fails += 1
print(f'multiplicativity: {passes} passes, {fails} fails (out of 50)')

## 5. `det(Aᵀ) = det(A)` and block-triangular determinants

Transpose preserves `det`. Block triangular matrices have `det M = det(A) · det(D)`.

In [ ]:
A = Matrix(QQ, [[2, 1, 3], [1, 4, 2], [3, 2, 5]])
print('det A  =', A.det(), '   det Aᵀ =', A.transpose().det())

# A 5x5 block upper triangular with 3x3 top-left block and 2x2 bottom-right.
A3 = Matrix(QQ, [[2, 1, 3], [1, 4, 2], [3, 2, 5]])   # det 3
D2 = Matrix(QQ, [[4, 1], [2, 3]])                    # det 4·3 − 1·2 = 10
zero = Matrix(QQ, 2, 3)
C    = Matrix(QQ, 3, 2, [1, 0, 0, 1, 1, 1])
M    = block_matrix([[A3, C], [zero, D2]])
show('M ='); show(M)
print('det M          =', M.det())
print('det A3 · det D2 =', A3.det() * D2.det(), '   (should match)')

## 6. Vandermonde symbolically: derive the product formula

Over the symbolic ring, Sage proves the identity `V(a, b, c) = (b−a)(c−a)(c−b)`.

In [ ]:
a, b, c = var('a b c')
V = Matrix(SR, [[1, 1, 1], [a, b, c], [a**2, b**2, c**2]])
show('V ='); show(V)
det_V = V.det()
print('det V (unexpanded) =', det_V)
print('det V (expanded)   =', det_V.expand())

identity = (b - a) * (c - a) * (c - b)
print('(b-a)(c-a)(c-b)    =', identity.expand())
print('match:', (det_V - identity).expand() == 0)

## 7. Cramer's Rule on a 3×3

Solve `2x + y + z = 4`, `x + 3y + z = 6`, `x + y + 2z = 5`. Expected: `(4/7, 9/7, 11/7)`.

In [ ]:
A = Matrix(QQ, [[2, 1, 1], [1, 3, 1], [1, 1, 2]])
bv = vector(QQ, [4, 6, 5])

det_A = A.det()
print(f'det A = {det_A}')

soln = []
for i in range(3):
    A_i = copy(A)
    A_i.set_column(i, bv)
    soln.append(A_i.det() / det_A)

show('Cramer:         x =', vector(QQ, soln))
show('solve_right:    x =', A.solve_right(bv))

## 8. Adjugate formula and `A · adj(A) = (det A) · I`

For `A = [[1,2,0],[0,1,3],[2,0,1]]`, `det A = 13` and the adjugate gives the exact inverse.

In [ ]:
def adjugate(M):
    '''Classical adjoint (adjugate): transpose of the cofactor matrix.'''
    n = M.nrows()
    C = Matrix(M.base_ring(), n, n)
    for i in range(n):
        for j in range(n):
            minor = M.delete_rows([i]).delete_columns([j])
            C[i, j] = (-1)**(i + j) * minor.det()
    return C.transpose()


A = Matrix(QQ, [[1, 2, 0], [0, 1, 3], [2, 0, 1]])
adj = adjugate(A)
show('adj(A) ='); show(adj)
print('det A =', A.det())
print('A · adj(A) =', A * adj)
print('(det A) · I =', A.det() * identity_matrix(QQ, 3))
print('match:', A * adj == A.det() * identity_matrix(QQ, 3))

Ainv_via_adj = adj / A.det()
print('A^-1 (via adj) =', Ainv_via_adj)
print('A^-1 (sage)    =', A.inverse())
print('match:', Ainv_via_adj == A.inverse())

## 9. Characteristic polynomial `det(A − λI)` — preview of Ch 9

Compute `det(A − λI)` for two matrices. The roots are the eigenvalues.

In [ ]:
R.<L> = PolynomialRing(QQ)

# (a) 2x2 from worked example E10
A = Matrix(QQ, [[4, 1], [2, 3]])
chi = (A - L * identity_matrix(QQ, 2)).det()
print('A =', A)
print('characteristic poly =', chi)
print('factored            =', chi.factor())
print('roots               =', chi.roots(QQ, multiplicities=True))

print()

# (b) 3x3 from exercise X10
A3 = Matrix(QQ, [[3, 1, 1], [1, 3, 1], [1, 1, 3]])
chi3 = (A3 - L * identity_matrix(QQ, 3)).det()
print('A3 =', A3)
print('characteristic poly =', chi3)
print('factored            =', chi3.factor())
print('roots               =', chi3.roots(QQ, multiplicities=True))

## 10. Exercises

Five exercises. Solutions are in the cell below each.

**A1.** Compute `det A` for `A = [[5,3,-1],[2,4,1],[0,1,2]]` by (i) cofactor expansion along column 3, (ii) Sage's `.det()`. They should agree.

In [ ]:
A = Matrix(QQ, [[5, 3, -1], [2, 4, 1], [0, 1, 2]])

# Cofactor along column 3 (index 2)
C13 = (-1)**(0+2) * Matrix(QQ, [[2, 4], [0, 1]]).det()
C23 = (-1)**(1+2) * Matrix(QQ, [[5, 3], [0, 1]]).det()
C33 = (-1)**(2+2) * Matrix(QQ, [[5, 3], [2, 4]]).det()
by_col3 = A[0,2]*C13 + A[1,2]*C23 + A[2,2]*C33
print('by column-3 cofactor expansion =', by_col3)
print('A.det()                        =', A.det())

**A2.** Let `A` be any `4 × 4` matrix with `det A = −3`. Without choosing a specific `A`, find `det(2A)`, `det(A⁻¹)`, `det(A²)`, and `det(Aᵀ A)`.

In [ ]:
# Pick any A with det = -3 to verify numerically.
A = Matrix(QQ, [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, -3]])
print(f'det A       = {A.det()}    (should be -3)')
print(f'det(2A)     = {(2*A).det()}    (expected 2^4 · (-3) = -48)')
print(f'det(A^-1)   = {A.inverse().det()}  (expected -1/3)')
print(f'det(A^2)    = {(A*A).det()}    (expected (-3)^2 = 9)')
print(f'det(A^T·A)  = {(A.transpose()*A).det()}    (expected (-3)^2 = 9)')

**A3.** Solve `x + 2y + z = 6`, `2x + y + 3z = 13`, `x − y + z = 2` using Cramer's Rule, and cross-check with `solve_right`.

In [ ]:
A = Matrix(QQ, [[1, 2, 1], [2, 1, 3], [1, -1, 1]])
bv = vector(QQ, [6, 13, 2])

det_A = A.det()
print(f'det A = {det_A}')

soln = []
for i in range(3):
    A_i = copy(A); A_i.set_column(i, bv)
    soln.append(A_i.det() / det_A)

print('Cramer:       x =', soln)
print('solve_right:  x =', A.solve_right(bv))

**A4.** For `A = [[3, 2], [1, 4]]`, compute the adjugate, verify `A · adj(A) = (det A) · I`, and read off `A⁻¹` from the formula.

In [ ]:
A = Matrix(QQ, [[3, 2], [1, 4]])
adj = adjugate(A)
print('det A      =', A.det())
print('adj(A)     =', adj)
print('A·adj(A)   =', A * adj)
print('(det A)·I  =', A.det() * identity_matrix(QQ, 2))
print('A^-1       =', adj / A.det())
print('sage A^-1  =', A.inverse())

**A5.** Find the eigenvalues of `A = [[2, 1, 0], [1, 2, 1], [0, 1, 2]]` by computing and factoring the characteristic polynomial.

In [ ]:
R.<L> = PolynomialRing(QQ)
A = Matrix(QQ, [[2, 1, 0], [1, 2, 1], [0, 1, 2]])
chi = (A - L * identity_matrix(QQ, 3)).det()
print('characteristic poly =', chi)
print('factored            =', chi.factor())
roots = chi.roots(AA, multiplicities=True)
print('roots (over AA)     =', roots)

## Takeaways

- **Never** use the cofactor recursion on anything larger than `n = 5` for computation — use `.det()` (Sage picks an efficient method).
- **Cramer's Rule** and the **adjugate** are theoretically gorgeous but computationally expensive.
- **Symbolic computation** (with `SR` and `var(...)`) lets you **prove** identities like the Vandermonde product formula — something NumPy can only sample at specific values.
- The **characteristic polynomial** `det(A − λI)` — computed here in exact rational arithmetic — is the bridge to Ch 9 (eigenvalues, diagonalization, dynamical systems).
